In [ ]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt

In [ ]:
data=pd.read_csv('/content/mnist_train - new.csv')

In [ ]:
import numpy as np
import requests

categories = [
    "apple","banana","car","airplane","bicycle",
    "cat","dog","fish","tree","house",
    "clock","star","cup","chair","hammer",
    "book","sun","moon","cloud","pizza"
]

base_url = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

for category in categories:
    print(f"Downloading {category}...")
    url = base_url + category + ".npy"
    response = requests.get(url)
    with open(category + ".npy", "wb") as f:
        f.write(response.content)

print("Download complete!")

In [ ]:
from sklearn.utils import shuffle
import numpy as np

samples_per_class = 20000

X = []
y = []

for label, category in enumerate(categories):
    data = np.load(category + ".npy")
    data = data[:samples_per_class]

    X.append(data)
    y.append(np.full(samples_per_class, label))

X = np.concatenate(X)   # (400000, 784)
y = np.concatenate(y)   # (400000,)

X, y = shuffle(X, y, random_state=42)

# Normalize
X = X / 255

# Transpose for NN
X = X.T   # (784, 400000)

# Split
X_dev = X[:, :1000]
Y_dev = y[:1000]

X_train = X[:, 1000:]
Y_train = y[1000:]

_, m_train = X_train.shape

In [ ]:
data = shuffle(data, random_state=42)

In [ ]:
data = np.array(data)
m,n = data.shape
np.random.shuffle(data)

In [ ]:
def init_params():
  W1 = np.random.randn(128, 784) * 0.01
  b1 = np.zeros((128, 1))
  W2 = np.random.randn(20, 128) * 0.01
  b2 = np.zeros((20, 1))
  #W1 = np.random.rand(10, 784) - 0.5
  #b1 = np.random.rand(10,1) - 0.5
  #W2 = np.random.rand(20,10) - 0.5
  #b2 = np.random.rand(20,1) - 0.5
  return W1, b1, W2, b2

In [ ]:
def ReLU(Z):
  return np.maximum(Z,0)

def softmax(Z):
  A = np.exp(Z) / sum(np.exp(Z))
  return A

In [ ]:
print(ReLU(np.array([[-2, 3, -1, 5]])))
print(softmax(np.array([[2], [1], [0.1]])))

In [ ]:
def forward_prop(W1, b1, W2, b2, X):
  Z1 = W1.dot(X) + b1
  A1 = ReLU(Z1)
  Z2 = W2.dot(A1) + b2
  A2 = softmax(Z2)
  return Z1, A1, Z2, A2

In [ ]:
W1, b1, W2, b2 = init_params()

In [ ]:
Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_train)

print("Z1:", Z1.shape)
print("A1:", A1.shape)
print("Z2:", Z2.shape)
print("A2:", A2.shape)

In [ ]:
print("data shape:", data.shape)

In [ ]:
def ReLU_deriv(Z):
  return Z>0

def one_hot(Y):
  one_hot_Y = np.zeros((Y.size, Y.max() + 1))
  one_hot_Y[np.arange(Y.size),Y] = 1
  one_hot_Y=one_hot_Y.T
  return one_hot_Y

In [ ]:
def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

In [ ]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
  W1 = W1 - alpha * dW1
  b1 = b1 - alpha * db1
  W2 = W2 - alpha * dW2
  b2 = b2 - alpha * db2
  return W1, b1, W2, b2

In [ ]:
def get_predictions(A2):
  return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
  print(predictions, Y)
  return np.sum(predictions==Y)/Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2



In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.05, 500)

In [ ]:
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    return predictions

In [ ]:
correct = 0
total = 0

In [ ]:
def test_prediction(index, W1, b1, W2, b2):
    global correct, total
    current_image = X_dev[:, index, None]
    prediction = make_predictions(X_dev[:, index, None], W1, b1, W2, b2)
    label = Y_dev[index]

    pred_num = prediction[0]
    label_num = label

    print("Prediction number:", pred_num)
    print("Label number:", label_num)
    print("Prediction name:", categories[pred_num])
    print("Label name:", categories[label_num])

        # accuracy check
    if pred_num == label_num:
      correct += 1

    total += 1
    accuracy = correct / total * 100

    print("Running Accuracy:", round(accuracy, 2), "%")
    print("---------------------------")


    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

test_prediction(0, W1, b1, W2, b2)
test_prediction(1, W1, b1, W2, b2)
test_prediction(2, W1, b1, W2, b2)
test_prediction(3, W1, b1, W2, b2)

dev_predictions = make_predictions(X_dev, W1, b1, W2, b2)
get_accuracy(dev_predictions, Y_dev)



In [ ]:
import random

for _ in range(10):
    index = random.randint(0, X_dev.shape[1]-1)
    test_prediction(index, W1, b1, W2, b2)

In [ ]:
for i, c in enumerate(categories):
    print(i, c)

In [ ]:
print(categories)

In [ ]:
print(np.unique(Y_train))

In [ ]:
print(Y_train[:20])

In [ ]:
print(len(np.unique(Y_train)))

In [ ]:
print(X.shape)
print(y.shape)
print(np.unique(y))

In [ ]:
print("X shape:", X.shape)
print("Y shape:", y.shape)

print("Unique labels:", np.unique(y))
print("Number of classes:", len(np.unique(y)))

print("First 20 labels:", y[:20])